# [Tutorial 2](https://github.com/evolutionaryscale/esm/tree/main/cookbook/tutorials): Embedding with ESM C

In this notebook we will see how to embed a batch of sequences using ESM C, as well as explore its different layers

API KEY: 1oTyCoZCRGcWg0e6EZBa3S

# Imports

# Set up Forge client for ESM C

Grab a token from [the Forge console](https://forge.evolutionaryscale.ai/console) and add it below. Note that your token is like a password for your account and you should take care to protect it. For this reason it is recommended to frequently create a new token and delete old, unused ones. It is also recommended to paste the token directly into an environment variable or use a utility like `getpass` as shown below so tokens are not accidentally shared or checked into code repositories.

In [42]:
import torch
from transformers import AutoTokenizer, EsmModel


model_name = "../esm2_t12_35M_UR50D/"

# --- 1. Setup ---
#protein_sequence = "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmModel.from_pretrained(
    model_name,
    output_hidden_states=True 
)

model.to(device)
model.eval()

"""# --- 3. Tokenize and Run the Model ---
inputs = tokenizer(protein_sequence, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# The hidden states are now in the 'hidden_states' attribute
all_hidden_states = outputs.hidden_states

# --- 4. Inspect the Hidden States ---
print("--- Output ---")
print(f"Total number of layers returned: {len(all_hidden_states)}")
print(f"This includes the embedding layer + all 12 hidden layers.")

print("\nShape of each layer's output tensor:")
for i, layer_tensor in enumerate(all_hidden_states):
    if i == 0:
        print(f"Layer {i} (Embedding Layer): \t{layer_tensor.shape}")
    else:
        print(f"Layer {i} (Hidden Layer {i}): \t{layer_tensor.shape}")
        """
        

Some weights of EsmModel were not initialized from the model checkpoint at ../esm2_t12_35M_UR50D/ and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


'# --- 3. Tokenize and Run the Model ---\ninputs = tokenizer(protein_sequence, return_tensors="pt").to(device)\n\nwith torch.no_grad():\n    outputs = model(**inputs)\n\n# The hidden states are now in the \'hidden_states\' attribute\nall_hidden_states = outputs.hidden_states\n\n# --- 4. Inspect the Hidden States ---\nprint("--- Output ---")\nprint(f"Total number of layers returned: {len(all_hidden_states)}")\nprint(f"This includes the embedding layer + all 12 hidden layers.")\n\nprint("\nShape of each layer\'s output tensor:")\nfor i, layer_tensor in enumerate(all_hidden_states):\n    if i == 0:\n        print(f"Layer {i} (Embedding Layer): \t{layer_tensor.shape}")\n    else:\n        print(f"Layer {i} (Hidden Layer {i}): \t{layer_tensor.shape}")\n        '

In [61]:
def embed_sequence(sequence, model, tokenizer):
    inputs = tokenizer(sequence, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        
    print(f"Outputs from model: {outputs}")
    print(f"Outputs hidden states: {outputs.hidden_states}")
    all_hidden_states = outputs.hidden_states
    return all_hidden_states


def batch_embed(sequences, model, tokenizer):
    results = []
    for seq in sequences:
        hidden_states = embed_sequence(seq, model, tokenizer)
        results.append(hidden_states)
    return results

In [63]:
!wget --no-check-certificate "https://docs.google.com/uc?export=download&id=1SpOkL11MJxIgy99dqufvUNJuCiuhxuyg" -O adk.csv

--2025-09-11 12:56:53--  https://docs.google.com/uc?export=download&id=1SpOkL11MJxIgy99dqufvUNJuCiuhxuyg
Resolving docs.google.com (docs.google.com)... 172.253.122.102, 172.253.122.100, 172.253.122.113, ...
Connecting to docs.google.com (docs.google.com)|172.253.122.102|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1SpOkL11MJxIgy99dqufvUNJuCiuhxuyg&export=download [following]
--2025-09-11 12:56:53--  https://drive.usercontent.google.com/download?id=1SpOkL11MJxIgy99dqufvUNJuCiuhxuyg&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.251.163.132
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.251.163.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 43132 (42K) [application/octet-stream]
Saving to: 'adk.csv'

adk.csv             100%[===================>]  42.12K  --.-KB/s    in 0.02s   

2025-09-11 12:56

In [64]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

adk_path = "adk.csv"
df = pd.read_csv(adk_path)
df = df[["org_name", "sequence", "lid_type", "temperature"]]
df = df[df["lid_type"] != "other"]  # drop one structural class for simplicity

In [46]:
print(df.head())
# get the sequence string from the first row
sequence = df.iloc[0]["sequence"]
print(sequence, len(sequence))
# print the number of rows
print(len(df))
## Height Width
## 172  * 194 

# only get the first 2 entries for testing
sequences = df["sequence"].tolist()


                   org_name  \
0  azorhizobium_caulinodans   
1   methylocella_silvestris   
2       bartonella_henselae   
3      sorangium_cellulosum   
4    elusimicrobium_minutum   

                                            sequence lid_type  temperature  
0  MRLVLLGPPGAGKGTQALRLVQRHGIVQLSTGDMLRAAVAAGTPVG...  lidless           30  
1  MRLVFLGPPGAGKGTQSTRLQQKFNIPQLSTGDMLRAAVKAGTPVG...  lidless           23  
2  MRIVLLGPPGAGKGTQAKMLCEEYHIPQLSTGDMLREVIRRETEIG...  lidless           37  
3  MILVLVGPPGAGKGTQAKLLCARFGIPQISTGDMLREAKRSGTLEK...  lidless           28  
4  MIIVLLGAPGAGKGTQSVLVAEKYGLKHISTGDLLREEIANNTELG...     zinc           26  
MRLVLLGPPGAGKGTQALRLVQRHGIVQLSTGDMLRAAVAAGTPVGLKAKAVMESGGLVSDEIVIGIIAERLDQPDARKGFILDGFPRTVAQADALDKLLADKGLKLDAVIELKVDQAKLLDRILNRAAEAKAKGEPVRKDDDPEVFKTRLEAYNRDTAVVAPYYSARGQLQQIDGMAPIEKVTQAIDSILETA 194
172


In [65]:
# You may see some error messages due to rate limits on each Forge account,
# but this will retry until the embedding job is complete
# This may take a few minutes to run
outputs = batch_embed(sequences, model, tokenizer)

Outputs from model: BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.2930, -0.2796,  0.0154,  ..., -0.5472,  0.0634,  0.0607],
         [-0.6453, -0.1398, -0.5287,  ..., -0.1098, -0.0972, -0.3656],
         [ 0.1331, -0.5902,  0.2923,  ...,  0.1142,  0.0023, -0.1070],
         ...,
         [ 0.1328, -0.2026,  0.1835,  ...,  0.1590, -0.1127, -0.3547],
         [ 0.2298, -0.1060, -0.1677,  ..., -0.4216, -0.3477, -0.3239],
         [-0.3886,  0.2354,  0.0985,  ..., -0.1719, -0.0942, -0.7498]]],
       device='mps:0'), pooler_output=tensor([[ 1.8024e-01,  7.0527e-02, -1.1186e-01,  1.8585e-01,  2.2603e-01,
         -1.3223e-02, -1.6697e-01,  6.7800e-02,  1.9598e-01,  5.3836e-02,
         -2.3565e-01,  8.0581e-02,  3.1810e-01,  1.8195e-01,  1.1194e-02,
         -3.3242e-02, -1.1750e-01,  2.0102e-01,  7.7354e-02,  2.1993e-02,
          1.5527e-01,  1.3184e-01,  9.0012e-02, -8.0136e-02,  2.7194e-01,
         -2.3682e-02, -3.6169e-01, -6.5771e-02, -1.6406e-02,  1.510

KeyboardInterrupt: 

In [60]:
#print(outputs)
print(f"Output shape: {len(outputs)}")
print(f"Single output shape: {outputs[0][0].shape}")
# convert the torch.size([1, 196, 480]) to a 2d array


for output in outputs:
    print(output[0].shape)

Output shape: 172
Single output shape: torch.Size([1, 196, 480])
torch.Size([1, 196, 480])
torch.Size([1, 195, 480])
torch.Size([1, 194, 480])
torch.Size([1, 217, 480])
torch.Size([1, 216, 480])
torch.Size([1, 199, 480])
torch.Size([1, 189, 480])
torch.Size([1, 194, 480])
torch.Size([1, 189, 480])
torch.Size([1, 183, 480])
torch.Size([1, 219, 480])
torch.Size([1, 221, 480])
torch.Size([1, 221, 480])
torch.Size([1, 222, 480])
torch.Size([1, 222, 480])
torch.Size([1, 219, 480])
torch.Size([1, 217, 480])
torch.Size([1, 217, 480])
torch.Size([1, 220, 480])
torch.Size([1, 216, 480])
torch.Size([1, 216, 480])
torch.Size([1, 213, 480])
torch.Size([1, 216, 480])
torch.Size([1, 226, 480])
torch.Size([1, 222, 480])
torch.Size([1, 247, 480])
torch.Size([1, 215, 480])
torch.Size([1, 216, 480])
torch.Size([1, 216, 480])
torch.Size([1, 216, 480])
torch.Size([1, 216, 480])
torch.Size([1, 216, 480])
torch.Size([1, 216, 480])
torch.Size([1, 216, 480])
torch.Size([1, 217, 480])
torch.Size([1, 216, 480])

In [54]:
import torch

# we'll summarize the embeddings using their mean across the sequence dimension
# which allows us to compare embeddings for sequences of different lengths
print(type(outputs))

all_mean_embeddings = [
    torch.mean(outputs, dim=-2).squeeze() for output in outputs
]

# now we have a list of tensors of [num_layers, hidden_size]
print("embedding shape [num_layers, hidden_size]:", all_mean_embeddings[0].shape)

<class 'list'>


TypeError: mean() received an invalid combination of arguments - got (list, dim=int), but expected one of:
 * (Tensor input, *, torch.dtype dtype = None, Tensor out = None)
 * (Tensor input, tuple of ints dim, bool keepdim = False, *, torch.dtype dtype = None, Tensor out = None)
 * (Tensor input, tuple of names dim, bool keepdim = False, *, torch.dtype dtype = None, Tensor out = None)


# Examine the performance of different layer embeddings

For this example, we're going to use PCA to visualize whether the embeddings separate our proteins by their structural class. To assess the quality of our PCA, we fit a K means classifier with three clusters, corresponding to the three structural classes of our enzyme, and compute the [rand index](https://en.wikipedia.org/wiki/Rand_index), a measure of the quality of the clustering.

In [21]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

N_KMEANS_CLUSTERS = 3

In [22]:
def plot_embeddings_at_layer(all_mean_embeddings: torch.Tensor, layer_idx: int):
    stacked_mean_embeddings = torch.stack(
        [embedding[layer_idx, :] for embedding in all_mean_embeddings]
    ).to(torch.float32).numpy()

    # project all the embeddings to 2D using PCA
    pca = PCA(n_components=2)
    pca.fit(stacked_mean_embeddings)
    projected_mean_embeddings = pca.transform(stacked_mean_embeddings)

    # compute kmeans purity as a measure of how good the clustering is
    kmeans = KMeans(n_clusters=N_KMEANS_CLUSTERS, random_state=0).fit(
        projected_mean_embeddings
    )
    rand_index = adjusted_rand_score(df["lid_type"], kmeans.labels_)

    # plot the clusters
    plt.figure(figsize=(4, 4))
    sns.scatterplot(
        x=projected_mean_embeddings[:, 0],
        y=projected_mean_embeddings[:, 1],
        hue=df["lid_type"],
    )
    plt.title(
        f"PCA of mean embeddings at layer {layer_idx}.\nRand index: {rand_index:.2f}"
    )
    plt.xlabel("PC 1")
    plt.ylabel("PC 2")
    plt.show()

In [24]:
#plot_embeddings_at_layer(all_mean_embeddings, layer_idx=30)
#plot_embeddings_at_layer(all_mean_embeddings, layer_idx=12)

plot_embeddings_at_layer([all_mean_embeddings, 0], layer_idx=0)

TypeError: list indices must be integers or slices, not tuple

We see that the top principal components of layer 12 separate structural classes better than that of layer 30. Embed away! And keep in mind that different layers may be better or worse for your particular use-case.